# Chia bảng `vqa` thành 5 range `vqa_id` cân bằng để annotate

Notebook này làm 4 việc:

1. Đọc dữ liệu từ **Supabase** (REST API) hoặc fallback từ file CSV cục bộ.  
2. Lọc dữ liệu theo nhu cầu annotate, ví dụ:
   - chỉ lấy `is_drop = False`
   - chỉ lấy `is_checked = False`
3. Sắp xếp theo `vqa_id` tăng dần.
4. Chia thành **5 đoạn liên tiếp theo `vqa_id`** sao cho **số lượng dòng giữa các đoạn gần bằng nhau nhất**.

Kết quả cuối cùng gồm:

- Bảng `assignment_ranges.csv`: mỗi thành viên một range `start_vqa_id` → `end_vqa_id`
- Bảng `vqa_with_member_assignment.csv`: mỗi dòng VQA được gán cho một `member_id`
- 5 file con: `member_1_vqa.csv` ... `member_5_vqa.csv`


In [ ]:
# Nếu cần, bỏ comment dòng dưới để cài thêm package ngay trong notebook
# !pip install pandas requests python-dotenv

import os
from pathlib import Path

import pandas as pd

try:
    import requests
except ImportError:
    raise ImportError("Thiếu package requests. Chạy: !pip install requests")

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

## 1) Cấu hình

- Nếu muốn đọc trực tiếp từ Supabase: đặt `USE_SUPABASE = True`
- Nếu muốn test nhanh bằng CSV: đặt `USE_SUPABASE = False`
- Mặc định notebook đang để:
  - chỉ lấy bản ghi **chưa check**
  - chỉ lấy bản ghi **không bị drop**


In [ ]:
USE_SUPABASE = True

# --- Supabase config ---
SUPABASE_URL = os.getenv("SUPABASE_URL", "").rstrip("/")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")  # service role key hoặc anon key có quyền đọc
TABLE_NAME = "vqa"
ID_COL = "vqa_id"
SELECT_COLS = "vqa_id,image_id,qtype,question,choice_a,choice_b,choice_c,choice_d,answer,is_checked,is_drop,created_at,updated_at"
PAGE_SIZE = 1000

# --- Fallback CSV config ---
CSV_FALLBACK_PATH = Path("vqa_rows.csv")

# --- Assignment config ---
N_MEMBERS = 5
FILTER_IS_DROP = False      # đặt None nếu muốn lấy tất cả
FILTER_IS_CHECKED = False   # đặt None nếu muốn lấy tất cả
OUTPUT_DIR = Path("outputs_vqa_assignment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "USE_SUPABASE": USE_SUPABASE,
    "TABLE_NAME": TABLE_NAME,
    "ID_COL": ID_COL,
    "N_MEMBERS": N_MEMBERS,
    "FILTER_IS_DROP": FILTER_IS_DROP,
    "FILTER_IS_CHECKED": FILTER_IS_CHECKED,
    "OUTPUT_DIR": str(OUTPUT_DIR.resolve()),
})

## 2) Hàm đọc dữ liệu từ Supabase và chia range

In [ ]:
def build_supabase_headers(api_key: str, want_count: bool = False) -> dict:
    headers = {
        "apikey": api_key,
        "Authorization": f"Bearer {api_key}",
    }
    if want_count:
        headers["Prefer"] = "count=exact"
    return headers


def fetch_supabase_table(
    supabase_url: str,
    api_key: str,
    table_name: str,
    select_cols: str = "*",
    id_col: str = "vqa_id",
    page_size: int = 1000,
    filter_is_drop=None,
    filter_is_checked=None,
) -> pd.DataFrame:
    if not supabase_url or not api_key:
        raise ValueError("Thiếu SUPABASE_URL hoặc SUPABASE_KEY.")

    endpoint = f"{supabase_url}/rest/v1/{table_name}"
    offset = 0
    all_rows = []

    while True:
        params = {
            "select": select_cols,
            "order": f"{id_col}.asc",
            "limit": page_size,
            "offset": offset,
        }

        if filter_is_drop is not None:
            params["is_drop"] = f"eq.{str(filter_is_drop).lower()}"

        if filter_is_checked is not None:
            params["is_checked"] = f"eq.{str(filter_is_checked).lower()}"

        resp = requests.get(
            endpoint,
            headers=build_supabase_headers(api_key),
            params=params,
            timeout=60,
        )
        resp.raise_for_status()

        batch = resp.json()
        if not batch:
            break

        all_rows.extend(batch)
        print(f"Đã tải {len(all_rows):,} dòng...")
        if len(batch) < page_size:
            break

        offset += page_size

    df = pd.DataFrame(all_rows)
    if df.empty:
        return df

    if id_col not in df.columns:
        raise KeyError(f"Không tìm thấy cột {id_col} trong dữ liệu Supabase.")

    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df = df.dropna(subset=[id_col]).copy()
    df[id_col] = df[id_col].astype("int64")
    df = df.sort_values(id_col, kind="stable").reset_index(drop=True)
    return df


def load_vqa_data(
    use_supabase: bool,
    csv_fallback_path: Path,
    supabase_url: str,
    api_key: str,
    table_name: str,
    select_cols: str,
    id_col: str,
    page_size: int,
    filter_is_drop=None,
    filter_is_checked=None,
) -> pd.DataFrame:
    if use_supabase:
        df = fetch_supabase_table(
            supabase_url=supabase_url,
            api_key=api_key,
            table_name=table_name,
            select_cols=select_cols,
            id_col=id_col,
            page_size=page_size,
            filter_is_drop=filter_is_drop,
            filter_is_checked=filter_is_checked,
        )
    else:
        if not csv_fallback_path.exists():
            raise FileNotFoundError(f"Không tìm thấy file CSV fallback: {csv_fallback_path}")
        df = pd.read_csv(csv_fallback_path)
        if id_col not in df.columns:
            raise KeyError(f"CSV không có cột {id_col}")
        df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
        df = df.dropna(subset=[id_col]).copy()
        df[id_col] = df[id_col].astype("int64")

        if filter_is_drop is not None and "is_drop" in df.columns:
            df = df[df["is_drop"] == filter_is_drop].copy()

        if filter_is_checked is not None and "is_checked" in df.columns:
            df = df[df["is_checked"] == filter_is_checked].copy()

        df = df.sort_values(id_col, kind="stable").reset_index(drop=True)

    if df.empty:
        raise ValueError("Không có dữ liệu sau khi đọc/lọc.")

    return df


def split_into_equal_count_ranges(df: pd.DataFrame, id_col: str = "vqa_id", n_groups: int = 5):
    if df.empty:
        raise ValueError("DataFrame rỗng.")
    if n_groups <= 0:
        raise ValueError("n_groups phải > 0.")
    if len(df) < n_groups:
        raise ValueError(f"Số dòng ({len(df)}) nhỏ hơn số nhóm ({n_groups}).")

    df = df.sort_values(id_col, kind="stable").reset_index(drop=True).copy()

    total = len(df)
    base_size = total // n_groups
    remainder = total % n_groups

    group_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_groups)]

    ranges = []
    assigned_frames = []
    start_idx = 0

    for group_idx, group_size in enumerate(group_sizes, start=1):
        end_idx = start_idx + group_size - 1
        chunk = df.iloc[start_idx:end_idx + 1].copy()

        start_vqa_id = int(chunk[id_col].iloc[0])
        end_vqa_id = int(chunk[id_col].iloc[-1])

        ranges.append({
            "member_id": group_idx,
            "start_row_idx": start_idx,
            "end_row_idx": end_idx,
            "start_vqa_id": start_vqa_id,
            "end_vqa_id": end_vqa_id,
            "row_count": len(chunk),
        })

        chunk["member_id"] = group_idx
        assigned_frames.append(chunk)

        start_idx = end_idx + 1

    ranges_df = pd.DataFrame(ranges)
    assigned_df = pd.concat(assigned_frames, ignore_index=True)

    # Validation
    assert len(assigned_df) == len(df), "Thiếu hoặc dư dòng sau khi gán nhóm."
    assert assigned_df[id_col].is_monotonic_increasing, "assigned_df không còn sort theo vqa_id."
    assert ranges_df["row_count"].sum() == len(df), "Tổng row_count không khớp."

    return ranges_df, assigned_df


def summarize_ranges(ranges_df: pd.DataFrame) -> pd.DataFrame:
    out = ranges_df.copy()
    out["range_label"] = out.apply(
        lambda x: f'vqa_id {x["start_vqa_id"]} → {x["end_vqa_id"]}',
        axis=1,
    )
    return out[["member_id", "range_label", "start_vqa_id", "end_vqa_id", "row_count"]]

## 3) Đọc dữ liệu

In [ ]:
df_vqa = load_vqa_data(
    use_supabase=USE_SUPABASE,
    csv_fallback_path=CSV_FALLBACK_PATH,
    supabase_url=SUPABASE_URL,
    api_key=SUPABASE_KEY,
    table_name=TABLE_NAME,
    select_cols=SELECT_COLS,
    id_col=ID_COL,
    page_size=PAGE_SIZE,
    filter_is_drop=FILTER_IS_DROP,
    filter_is_checked=FILTER_IS_CHECKED,
)

print(f"Số dòng sau khi đọc/lọc: {len(df_vqa):,}")
display(df_vqa.head())

## 4) Chia thành 5 nhóm có số lượng gần bằng nhau

In [ ]:
ranges_df, assigned_df = split_into_equal_count_ranges(
    df=df_vqa,
    id_col=ID_COL,
    n_groups=N_MEMBERS,
)

summary_df = summarize_ranges(ranges_df)
display(summary_df)

print("Độ lệch số dòng giữa nhóm lớn nhất và nhỏ nhất:",
      int(ranges_df["row_count"].max() - ranges_df["row_count"].min()))

## 5) Kiểm tra nhanh từng nhóm

In [ ]:
for member_id, sub in assigned_df.groupby("member_id", sort=True):
    print(
        f"Member {member_id}: "
        f"{len(sub):,} dòng | "
        f"vqa_id {sub[ID_COL].min()} -> {sub[ID_COL].max()}"
    )

## 6) Xuất file CSV để chia cho từng thành viên

In [ ]:
ranges_path = OUTPUT_DIR / "assignment_ranges.csv"
assigned_path = OUTPUT_DIR / "vqa_with_member_assignment.csv"

summary_df.to_csv(ranges_path, index=False, encoding="utf-8-sig")
assigned_df.to_csv(assigned_path, index=False, encoding="utf-8-sig")

member_files = []
for member_id, sub in assigned_df.groupby("member_id", sort=True):
    out_path = OUTPUT_DIR / f"member_{member_id}_vqa.csv"
    sub.to_csv(out_path, index=False, encoding="utf-8-sig")
    member_files.append(str(out_path))

print("Đã lưu:")
print("-", ranges_path)
print("-", assigned_path)
for p in member_files:
    print("-", p)

## 7) Nếu bạn chỉ muốn copy-paste kết quả range

Cell này in ra 5 dòng gọn để giao việc nhanh.


In [ ]:
for row in summary_df.itertuples(index=False):
    print(
        f"Thành viên {row.member_id}: "
        f"vqa_id từ {row.start_vqa_id} đến {row.end_vqa_id} "
        f"({row.row_count} câu)"
    )

## Ghi chú

- Nếu `vqa_id` bị khuyết số, notebook vẫn hoạt động bình thường.  
  Mỗi nhóm là **một đoạn liên tiếp trong thứ tự `vqa_id` hiện có**, không phải chia đều theo độ rộng số học.
- Nếu bạn muốn chia theo **toàn bộ bảng**, đổi:
  - `FILTER_IS_DROP = None`
  - `FILTER_IS_CHECKED = None`
- Nếu muốn đổi số người annotate, chỉ cần sửa `N_MEMBERS`.
